# S1 equities — FTMO 2-Step challenge select

Compare **FTMO Challenge 2-Step** (not 1-step) pass probability and **economic EV** (fee, payout split, retries). This is **not** $P(\mathrm{beat\ SPY})$ — that lives under `06_risk/notebooks/` (`02_ev_vs_spy.ipynb`).

Fees, account size, profit split, and horizon $H$ are **inputs** (not official FTMO prices). Official 2-step rules encoded here have no calendar time cap; $H$ is the simulation window. `incomplete` at $H$ is not a rule violation.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.leverage.policy import k_fair_from_artifact
from risk.analytics.leverage.artifacts import load_leverage_artifact
from risk.analytics.monte_carlo.loaders import find_repo_root, load_sealed_s1, load_sealed_s2
from risk.analytics.prop_firm.registry import CHALLENGES, make_challenge
from risk.analytics.prop_firm.report import (
    binding_mix,
    fair_vs_ftmo_row,
    leverage_ev_grid,
    run_challenge_select,
    suggest_challenge_leverage,
)
from risk.analytics.prop_firm.s1_calendar import weekly_to_weekday_returns
from risk.analytics.prop_firm.plots import failure_mix_figure, leverage_heatmap_figure, retries_hist_figure

ROOT = find_repo_root(ROOT)
print("ROOT", ROOT)
print("registered challenges", sorted(CHALLENGES))

SLEEVE = 's1'
FIRM_KEY = 'ftmo.2step'


## 1. Data Loading


In [ ]:
RETURNS_WEEKLY = load_sealed_s1(ROOT)
# S1 weekly -> weekday expansion: full week PnL on the index date (Monday),
# other weekdays 0. Does not divide by 5. Conservative for 5% daily loss.
RETURNS = weekly_to_weekday_returns(RETURNS_WEEKLY)
DEFAULT_H = 60
DEFAULT_HF = 40
ART = load_leverage_artifact(ROOT, "s1")
K_FAIR = k_fair_from_artifact(ART, RETURNS_WEEKLY, periods_per_year=52.0)
print("weekly bars", len(RETURNS_WEEKLY), "weekday bars", len(RETURNS))
print("leverage artifact", None if ART is None else ART.get("target_ann_vol"))
print("k_fair (VT-equivalent at this sample vol)", K_FAIR)
print(RETURNS.head(10))


## 2. Challenge selector

v1 registry: `ftmo.2step.challenge` / `.verification` / `.funded`. Later firms register new classes without rewriting this notebook.

**S1 approximation:** sealed returns are weekly. Each week is expanded to weekdays with the full weekly return on Monday and zeros elsewhere. Intra-week mark-to-market is unknown; concentrating PnL on the decision day is conservative for FTMO daily loss. ~1 trading day is counted per week. If a daily S1 equity parquet appears later, swap this loader only.


In [ ]:
print(make_challenge("ftmo.2step.challenge").name())
print("challenge target", make_challenge("ftmo.2step.challenge").profit_target_frac())
print("verification target", make_challenge("ftmo.2step.verification").profit_target_frac())
print("funded target", make_challenge("ftmo.2step.funded").profit_target_frac())


## 3. Pass rates & economic EV

Suggested challenge $k$ is `min(k_fair, k_ftmo)` from the EV/day grid under daily/max-loss fail-rate caps. It is shown in the headline, not only the slider.


In [ ]:
PACK = {"headline": None, "results": None, "suggestion": None, "grid": None}

GRID_KS = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
MAX_P_FAIL_DAILY = 0.40
MAX_P_FAIL_MAX = 0.30

def _grid(n_simulations, horizon, horizon_funded, initial_capital, fee, profit_split, mean_block_length):
    return leverage_ev_grid(
        RETURNS,
        GRID_KS,
        n_simulations=int(n_simulations),
        horizon=int(horizon),
        horizon_funded=int(horizon_funded),
        initial_capital=float(initial_capital),
        fee=float(fee),
        profit_split=float(profit_split),
        mean_block_length=float(mean_block_length),
        random_seed=0,
    )

def run(n_simulations, horizon, horizon_funded, leverage, initial_capital, fee, profit_split, mean_block_length):
    pack = run_challenge_select(
        RETURNS,
        n_simulations=int(n_simulations),
        horizon=int(horizon),
        horizon_funded=int(horizon_funded),
        leverage=float(leverage),
        initial_capital=float(initial_capital),
        fee=float(fee),
        profit_split=float(profit_split),
        mean_block_length=float(mean_block_length),
        random_seed=0,
    )
    PACK.update(pack)
    head = pack["headline"].copy()
    sug = PACK.get("suggestion") or {}
    for k in ("k_fair", "k_ftmo", "k_suggested"):
        if k in sug:
            head[k] = sug[k]
    display(head.to_frame("value"))
    print("do_not_take (EV<=0 or lower CI<=0):", bool(pack["headline"]["do_not_take"]))
    print("k_suggested = min(k_fair, k_ftmo)", sug.get("k_suggested"), "k_fair", sug.get("k_fair"), "k_ftmo", sug.get("k_ftmo"))
    return pack

# Coarse grid first so the headline can show a suggested k, then run once at that k.
grid = _grid(80, DEFAULT_H, DEFAULT_HF, 100000.0, 540.0, 0.8, 10.0)
sug = suggest_challenge_leverage(
    grid, k_fair=K_FAIR, max_p_fail_daily=MAX_P_FAIL_DAILY, max_p_fail_max=MAX_P_FAIL_MAX
)
PACK["grid"] = grid
PACK["suggestion"] = sug
display(pd.Series(sug, name="k_policy").to_frame("value"))
K_SUGGESTED = float(sug["k_suggested"]) if sug.get("k_suggested") == sug.get("k_suggested") else 1.0
if not (K_SUGGESTED == K_SUGGESTED):
    K_SUGGESTED = 1.0
pack = run(300, DEFAULT_H, DEFAULT_HF, K_SUGGESTED, 100000.0, 540.0, 0.8, 10.0)
try:
    import ipywidgets as w
    ui = w.interactive(
        run,
        n_simulations=w.IntSlider(min=50, max=1500, value=300, step=50, description="n_sim"),
        horizon=w.IntSlider(min=10, max=180, value=DEFAULT_H, step=5, description="H eval"),
        horizon_funded=w.IntSlider(min=10, max=180, value=DEFAULT_HF, step=5, description="H funded"),
        leverage=w.FloatSlider(min=0.25, max=3.0, value=K_SUGGESTED, step=0.25, description="k"),
        initial_capital=w.Dropdown(options=[10000.0, 25000.0, 50000.0, 100000.0, 200000.0], value=100000.0, description="size"),
        fee=w.FloatSlider(min=0.0, max=2000.0, value=540.0, step=20.0, description="fee"),
        profit_split=w.FloatSlider(min=0.5, max=0.9, value=0.8, step=0.05, description="split"),
        mean_block_length=w.FloatSlider(min=2.0, max=25.0, value=10.0, step=1.0, description="block L"),
    )
    display(ui)
except Exception as exc:
    print("ipywidgets unavailable (%s); default run already executed" % exc)


## 4. Evaluation

$P(\mathrm{challenge})$, $P(\mathrm{verification})$, $P(\mathrm{both})$, median/p90 days-to-pass, failure mix, economic EV with percentile CI and $P(\mathrm{EV}\le 0)$, EV/day, do-not-take. Optimize **EV/day**, not raw pass rate.


In [ ]:
if PACK.get("results") is not None:
    mix = binding_mix(PACK["results"], phase="chal")
    display(mix.to_frame("n"))
    failure_mix_figure(mix).show()
    retries_hist_figure(PACK["retries_until_pass"]).show()
else:
    print("Run the selector in section 3 first.")
